## Import Library

In [8]:
import requests
import trafilatura
import pandas as pd
import re, time
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

print(f"trafilatura {trafilatura.__version__} | pandas {pd.__version__} | requests {requests.__version__} OK")

trafilatura 2.2.0 | pandas 3.0.5 | requests 2.34.2 OK


## Konfigurasi 2 Tema

In [9]:
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept-Language": "id-ID,id;q=0.9,en;q=0.8",
}

TOPICS = [
    {"tema": "finance",  "url": "https://finance.detik.com/"},
    {"tema": "olahraga", "url": "https://sport.detik.com/"},
]

MAX_PER_TEMA = 100
OUTPUT_CSV = "detik_berita.csv"
pd.DataFrame(TOPICS)

,tema,url
0,finance,https://finance.detik.com/
1,olahraga,https://sport.detik.com/


## Main — Crawling 100 berita per tema 

Target `100/100` per tema → total `200` artikel.

In [10]:
all_data = []
for topic in TOPICS:
    print(f"\n=== {topic['tema'].upper()} | {topic['url']} ===", flush=True)
    html = fetch(topic["url"])
    if not html:
        continue
    links = get_article_links(html, topic["url"])
    print(f"  Ditemukan {len(links)} link", flush=True)
    candidates = links[:MAX_PER_TEMA*4]
    # target 100 -> butuh ~400 kandidat, crawl indeks 2..15
    if len(candidates) < MAX_PER_TEMA*3:
        for page in range(2, 16):
            for extra_url in [topic["url"].rstrip("/") + f"/indeks?page={page}", topic["url"] + f"?page={page}"]:
                extra_html = fetch(extra_url)
                if extra_html:
                    extra_links = get_article_links(extra_html, topic["url"])
                    new = 0
                    for l in extra_links:
                        if l not in candidates:
                            candidates.append(l)
                            new += 1
                    if new > 0:
                        print(f"  + halaman {page} ({extra_url[-20:]}): +{new} -> total {len(candidates)}", flush=True)
                        break
            if len(candidates) >= MAX_PER_TEMA*4:
                break
        print(f"  Setelah indeks tambahan: {len(candidates)} kandidat", flush=True)
    
    scraped = []
    with ThreadPoolExecutor(max_workers=5) as ex:
        futures = {ex.submit(scrape_article_trafilatura, link): link for link in candidates}
        for f in tqdm(as_completed(futures), total=len(futures), desc=f"{topic['tema']}"):
            link = futures[f]
            try:
                art = f.result()
            except Exception as e:
                print(f"    ERROR {link[-40:]}: {e}", flush=True)
                continue
            if art:
                art["tema"] = topic["tema"]
                scraped.append(art)
                print(f"    OK | {art['url'][-60:]}... ({len(art['isi_berita'])} char)", flush=True)
                if len(scraped) >= MAX_PER_TEMA:
                    for fut in futures:
                        fut.cancel()
                    break
            else:
                print(f"    SKIP ...{link[-50:]}", flush=True)
    
    scraped = scraped[:MAX_PER_TEMA]
    print(f"  -> Terkumpul {len(scraped)}/{MAX_PER_TEMA} untuk {topic['tema']}", flush=True)
    all_data.extend(scraped)
    time.sleep(0.3)

# deduplicate by url
seen = set()
unique = []
for a in all_data:
    if a["url"] not in seen:
        seen.add(a["url"])
        unique.append(a)
all_data = unique
print(f"\nTotal unik: {len(all_data)} artikel", flush=True)


=== FINANCE | https://finance.detik.com/ ===
  Ditemukan 52 link
  + halaman 2 (ik.com/indeks?page=2): +3 -> total 55
  + halaman 3 (ik.com/indeks?page=3): +17 -> total 72
  + halaman 4 (ik.com/indeks?page=4): +18 -> total 90
  + halaman 5 (ik.com/indeks?page=5): +20 -> total 110
  + halaman 6 (ik.com/indeks?page=6): +19 -> total 129
  + halaman 7 (ik.com/indeks?page=7): +18 -> total 147
  + halaman 8 (ik.com/indeks?page=8): +17 -> total 164
  + halaman 9 (ik.com/indeks?page=9): +20 -> total 184
  + halaman 10 (k.com/indeks?page=10): +20 -> total 204
  + halaman 11 (k.com/indeks?page=11): +20 -> total 224
  + halaman 12 (k.com/indeks?page=12): +17 -> total 241
  + halaman 13 (k.com/indeks?page=13): +20 -> total 261
  + halaman 14 (k.com/indeks?page=14): +19 -> total 280
  + halaman 15 (k.com/indeks?page=15): +13 -> total 293
  Setelah indeks tambahan: 293 kandidat


finance:   0%|          | 0/293 [00:00<?, ?it/s]

    OK | berita-ekonomi-bisnis/d-8654941/harga-emas-antam-makin-jatuh... (1733 char)


finance:   0%|          | 1/293 [00:01<05:58,  1.23s/it]

    OK | pm-singapura-sumbangkan-kenaikan-gaji-rp-24-m-selama-5-tahun... (2183 char)
    OK | dibuatkan-rekening-saldo-awal-rp-50-ribu-sumber-apbn-rp-11-t... (1221 char)
    OK | /bri-ditugaskan-buka-rekening-warga-ri-khusus-aceh-lewat-bsi... (1274 char)


...

    OK | agar-motogp-indonesia-2026-sukses-di-dalam-dan-luar-lintasan... (3304 char)
    OK | 7/motogp-2026-aprilia-kian-termotivasi-kalahkan-marc-marquez... (1618 char)
    OK | china-masters-2026-leo-daniel-kena-comeback-keok-di-16-besar... (1549 char)


olahraga:  30%|███       | 99/326 [00:16<00:32,  7.08it/s]

    OK | ik.com/moto-gp/d-8644946/francesco-bagnaia-di-titik-terendah... (1489 char)


olahraga:  30%|███       | 99/326 [00:16<00:38,  5.84it/s]


  -> Terkumpul 100/100 untuk olahraga

Total unik: 200 artikel


## Result — CSV + Preview

In [11]:
# Result CSV - 3 kolom Excel-friendly: id | isi_berita | tema (A=id, B=isi, C=tema)
import csv
# buat DataFrame dengan urutan sesuai request: A=id, B=isi_berita, C=tema
df = pd.DataFrame([{"id": idx, "isi_berita": a["isi_berita"].replace(";", ",").replace("\n"," ").replace("\r"," "), "tema": a["tema"]} for idx, a in enumerate(all_data, 1)])
df = df[["id", "isi_berita", "tema"]]  # urutan: A=id, B=isi_berita, C=tema
assert list(df.columns) == ["id", "isi_berita", "tema"], "Kolom harus 3: id, isi_berita, tema"
assert df.shape[1] == 3, "Jumlah kolom harus 3"
# cek isi tidak ada newline/; yang bikin Excel pindah cell/baris
assert not df["isi_berita"].str.contains(";").any(), "isi_berita masih ada ;"
# simpan dengan sep=; (Excel Indonesia) dan quote minimal agar 1 baris = 1 artikel
try:
    df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig", sep=";", quoting=csv.QUOTE_MINIMAL, lineterminator="\n")
    print(f"Saved -> {OUTPUT_CSV} : {df.shape[0]} rows x {df.shape[1]} cols (sep=';')", flush=True)
except PermissionError:
    alt = OUTPUT_CSV.replace(".csv", "_new.csv")
    df.to_csv(alt, index=False, encoding="utf-8-sig", sep=";", quoting=csv.QUOTE_MINIMAL, lineterminator="\n")
    print(f"[WARN] {OUTPUT_CSV} terkunci -> saved ke {alt} : {df.shape[0]} rows", flush=True)
print(f"Jumlah kolom: {df.shape[1]}", flush=True)
print(f"Daftar kolom:", flush=True)
print(f"  id         ", flush=True)
print(f"  isi_berita ", flush=True)
print(f"  tema       ", flush=True)
print(f"\nTotal artikel: {len(df)}", flush=True)
print("\nDistribusi tema (tabel):", flush=True)
tabel_tema = df["tema"].value_counts().to_frame("jumlah")
tabel_tema = tabel_tema.reindex([t for t in ["finance", "olahraga"] if t in tabel_tema.index])
display(tabel_tema)
for tema, jumlah in tabel_tema["jumlah"].items():
    print(f"  - {tema}: {jumlah} artikel", flush=True)


Saved -> detik_berita.csv : 200 rows x 3 cols (sep=';')
Jumlah kolom: 3
Daftar kolom:
  id         
  isi_berita 
  tema       

Total artikel: 200

Distribusi tema (tabel):


,jumlah
tema,
finance,100
olahraga,100


  - finance: 100 artikel
  - olahraga: 100 artikel
